In [1]:
import pandas as pd

from sklearn.model_selection import GridSearchCV

from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

from sklearn.model_selection import KFold

import mlflow

from src.data_preparation import clean_data
from src.feature_engineering import add_features
from src.preprocessing import build_preprocessor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/House_Prices.csv")

df = clean_data(df)
df = add_features(df)

X = df.drop(columns="SalePrice")
y = df["SalePrice"]

preprocessor = build_preprocessor(X)

In [3]:
gradient_boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

In [4]:
param_grid = {
    "model__n_estimators": [100, 200,300, 400, 500],
    "model__learning_rate": [0.02, 0.05, 0.07, 0.1, 0.2],
    "model__max_depth": [2, 3, 4, 5, 6],
}

In [5]:
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [6]:
grid_search = GridSearchCV(
    estimator=gradient_boosting_pipeline,
    param_grid=param_grid,
    cv=kfold,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
mlflow.set_tracking_uri(
    "sqlite:////home/elgmouri/projects/aqari/mlflow.db"
)
mlflow.set_experiment("Aqari Model Comparison")
with mlflow.start_run(run_name="Gradient Boosting GridSearchCV"):
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred_tuned = best_model.predict(X_test)
    mlflow.log_params(grid_search.best_params_)

    tuned_mae = mean_absolute_error(y_test, y_pred_tuned)
    tuned_rmse = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
    tuned_r2 = r2_score(y_test, y_pred_tuned)

    mlflow.log_metrics({
    "tuned_mae": tuned_mae,
    "tuned_rmse": tuned_rmse,
    "tuned_r2": tuned_r2,
    "best_cv_rmse": -grid_search.best_score_
})

In [9]:
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation RMSE:", -grid_search.best_score_)

Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 500}
Best cross-validation RMSE: 22754.932205023877


In [10]:
print("Tuned MAE:", tuned_mae)
print("Tuned RMSE:", tuned_rmse)
print("Tuned R²:", tuned_r2)

Tuned MAE: 14649.606907298923
Tuned RMSE: 20788.8428072725
Tuned R²: 0.9217600802459367


In [11]:
gradient_boosting_pipeline.fit(X_train, y_train)
y_pred_original = gradient_boosting_pipeline.predict(X_test)

original_mae = mean_absolute_error(y_test, y_pred_original)
original_rmse = np.sqrt(mean_squared_error(y_test, y_pred_original))
original_r2 = r2_score(y_test, y_pred_original)

In [12]:
print("Before tuning")
print("MAE:", original_mae)
print("RMSE:", original_rmse)
print("R²:", original_r2)
print("\nAfter tuning")
print("MAE:", tuned_mae)
print("RMSE:", tuned_rmse)
print("R²:", tuned_r2)

Before tuning
MAE: 14531.808750105632
RMSE: 20072.89440595186
R²: 0.9270563038889157

After tuning
MAE: 14649.606907298923
RMSE: 20788.8428072725
R²: 0.9217600802459367


On the same 20% holdout split (random_state=42), tuning decreased the quality , The original untuned Gradient Boosting performed better on this test set; the best cross-validation parameters did not improve holdout performance